# Object-Oriented Programming (OOP) in Python — Concepts & Code

This notebook introduces **just enough Object-Oriented Programming (OOP)** for a student heading toward Machine Learning. We keep it practical: no deep theory, no design patterns — only what you need to *read and write* the kind of code you'll see in NumPy, Pandas, and Scikit-learn (which are all built using classes and objects internally, and whose APIs — like `model.fit(X, y)` and `model.predict(X)` — are OOP in action).

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what a class and an object are, and why OOP is useful.
- Define a class with an `__init__` constructor and use `self` correctly.
- Distinguish instance attributes from class attributes.
- Write instance methods that operate on an object's data.
- Create multiple independent objects from the same class.
- Apply basic encapsulation conventions in Python.
- Add readable string representations using `__str__`.
- Use inheritance to reuse and extend behavior with `super()`.
- Recognize polymorphism in everyday Python code.
- Recognize the `fit` / `predict`-style OOP pattern used throughout Scikit-learn.

## Prerequisites
- Comfortable with Core Python: variables, data types, functions, dictionaries, lists (see `Core_Python_for_ML_Part_1.ipynb` and `Part_2.ipynb`).

## Companion notebook
Once you've worked through this notebook, open **`OOP_Practice_Exercises.ipynb`** to test yourself with hands-on tasks.

---

## Table of Contents
1. [What is OOP, and Why Does it Matter for ML?](#what-is-oop)
2. [Classes and Objects](#classes-objects)
3. [The `__init__` Constructor and `self`](#init-self)
4. [Instance Attributes vs Class Attributes](#attributes)
5. [Instance Methods](#methods)
6. [Creating Multiple Objects](#multiple-objects)
7. [Encapsulation Basics](#encapsulation)
8. [Readable Objects with `__str__`](#str-method)
9. [Inheritance](#inheritance)
10. [Method Overriding and `super()`](#overriding)
11. [Polymorphism](#polymorphism)
12. [ML Preview: The `fit` / `predict` Pattern](#ml-preview)
13. [Common Mistakes Recap](#common-mistakes)
14. [Summary](#summary)


<a id="what-is-oop"></a>
# 1. What is OOP, and Why Does it Matter for ML?

So far, you've written code using variables, functions, lists, and dictionaries — this is called **procedural programming**: a sequence of steps operating on separate pieces of data.

**Object-Oriented Programming (OOP)** is a different way of organizing code: instead of keeping data and the functions that operate on it separate, you bundle them together into a single unit called an **object**.

- A **class** is a *blueprint* for creating objects — it defines what data (attributes) and behavior (methods) the objects will have.
- An **object** (also called an *instance*) is a specific thing built from that blueprint.

### Why should an ML student care?
Every major ML/Data Science library is built on OOP, and you'll *use* objects constantly even before you write your own classes:

```python
model = LinearRegression()   # creating an object from a class
model.fit(X_train, y_train)  # calling a method on that object
predictions = model.predict(X_test)  # calling another method
```

Here, `model` is an **object**. `LinearRegression` is the **class**. `.fit()` and `.predict()` are **methods** — functions that belong to the object. Understanding OOP means you understand *why* Scikit-learn's API looks the way it does, instead of it feeling like magic.

<a id="classes-objects"></a>
# 2. Classes and Objects

Let's build the simplest possible class: one with no data and no behavior, just to see the syntax.

In [ ]:
class Student:
    pass   # 'pass' means "do nothing" -- a placeholder so the class has a body

# Creating an object (an "instance") of the Student class
student1 = Student()

print(student1)
print(type(student1))

**Output (yours will differ slightly):**
```
<__main__.Student object at 0x...>
<class '__main__.Student'>
```

Right now, `student1` doesn't hold any useful data. Let's fix that next.

<a id="init-self"></a>
# 3. The `__init__` Constructor and `self`

`__init__` is a special method that runs **automatically** whenever you create a new object. It's used to set up the object's initial data — this is called the **constructor**.

`self` refers to **the specific object being created or used**. It's always the first parameter of every instance method, though Python passes it automatically — you never type it yourself when *calling* a method.

In [ ]:
class Student:
    def __init__(self, name, age):
        # self.name and self.age are ATTRIBUTES -- data that belongs to this object
        self.name = name
        self.age = age

student1 = Student("Ali", 22)
print(student1.name)
print(student1.age)

**Common mistake:** Forgetting `self` as the first parameter of `__init__` (or any instance method). Without it, Python has no way to know *which* object's data you're setting, and you'll get a `TypeError` about the number of arguments.

<a id="attributes"></a>
# 4. Instance Attributes vs Class Attributes

- **Instance attributes** (like `self.name` above) belong to one specific object. Each object has its own copy.
- **Class attributes** are defined directly inside the class body (not inside `__init__`) and are **shared by all objects** of that class, unless a specific object overrides them.

In [1]:
class Student:
    school_name = "AIML Academy"   # class attribute -- shared by every Student object

    def __init__(self, name, age):
        self.name = name           # instance attribute -- unique per object
        self.age = age

student1 = Student("Ali", 22)
student2 = Student("Sara", 21)

print(student1.name, student1.school_name)
print(student2.name, student2.school_name)

# Changing an instance attribute affects ONLY that object:
student1.school_name = "Data Science Institute"
print(student1.school_name)   # changed
print(student2.school_name)   # unaffected -- still the original class attribute

Ali AIML Academy
Sara AIML Academy
Data Science Institute
AIML Academy


**Common mistake:** Using a **mutable** class attribute (like a list) expecting each object to get its own copy. In reality, all objects would *share* the same list, and modifying it through one object affects every object. Always initialize mutable data (lists, dicts) inside `__init__` as instance attributes instead.

<a id="methods"></a>
# 5. Instance Methods

A **method** is a function defined inside a class. It automatically receives `self`, giving it access to the object's own attributes.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks   # a list of marks

    def average(self):
        return sum(self.marks) / len(self.marks)

    def grade(self):
        avg = self.average()   # methods can call other methods via self
        if avg >= 90:
            return "A"
        elif avg >= 75:
            return "B"
        elif avg >= 60:
            return "C"
        else:
            return "F"

student1 = Student("Ali", [78, 85, 90])
print(student1.average())
print(student1.grade())

### ML Connection
This is exactly the shape of `model.fit(...)` and `model.predict(...)`: methods that use the object's own internal state (`self.marks` here, or a trained model's learned weights in Scikit-learn) to compute a result.

<a id="multiple-objects"></a>
# 6. Creating Multiple Objects

The whole point of a class is that you can create **many independent objects** from the same blueprint, each with their own data.

In [2]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):
        return sum(self.marks) / len(self.marks)


students = [
    Student("Ali", [78, 85, 90]),
    Student("Sara", [88, 92, 95]),
    Student("Ahmed", [55, 62, 58]),
]

for s in students:
    print(f"{s.name}: {s.average():.2f}")

Ali: 84.33
Sara: 91.67
Ahmed: 58.33


### ML Connection
> Notice how similar this is to the list-of-dictionaries pattern from the earlier Core Python notebooks (`students = [{"name": ..., "marks": [...]}]`). Classes are a more structured, self-contained alternative — each object bundles its data *and* the logic to process that data together.

<a id="encapsulation"></a>
# 7. Encapsulation Basics

**Encapsulation** means bundling data and the methods that operate on it together, and controlling how that data is accessed or modified from outside the object. Python doesn't enforce strict access control the way some languages do, but it uses **naming conventions**:

- `name` — public: intended to be used freely from outside the class.
- `_name` — "protected" by convention: a hint that this is for internal use, though Python doesn't actually stop you from accessing it.
- `__name` — "private" by convention: Python performs *name mangling* to make accidental access harder (not impossible).

In [5]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance   # convention: internal use, but still accessible

    def deposit(self, amount):
        if amount <= 0:
            print("Deposit amount must be positive.")
            return
        self.__balance += amount

    def get_balance(self):
        return self.__balance


account = BankAccount("Ali", 1000)
account.deposit(500)
print(account.get_balance())

# Still technically possible, but goes against convention:
account.balance = 999999
print(account.get_balance())

1500
1500


**Common mistake:** Directly modifying an attribute prefixed with `_` from outside the class (as shown above). It "works," but it breaks the intended design — always prefer the class's own methods (like `deposit()`) when they exist.

<a id="str-method"></a>
# 8. Readable Objects with `__str__`

By default, printing an object shows an unhelpful memory address (as you saw earlier). Defining a `__str__` method lets you control what `print(object)` displays.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):
        return sum(self.marks) / len(self.marks)

    def __str__(self):
        return f"Student(name={self.name}, average={self.average():.2f})"


student1 = Student("Ali", [78, 85, 90])
print(student1)          # now uses __str__ automatically

**Output:**
```
Student(name=Ali, average=84.33)
```

Methods like `__init__` and `__str__`, surrounded by double underscores, are called **"dunder" (double underscore) methods**. Python calls them automatically in specific situations. You don't need to memorize many of these — `__init__` and `__str__` are the two most useful for a beginner.

<a id="inheritance"></a>
# 9. Inheritance

**Inheritance** lets a new class (the **child** or **subclass**) reuse the attributes and methods of an existing class (the **parent** or **superclass**), while adding or changing behavior. This avoids rewriting shared logic.

In [8]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"Hi, I'm {self.name} and I'm {self.age} years old."


class Student(Person):          # Student inherits from Person
    def __init__(self, name, age, marks):
        super().__init__(name, age)   # call the parent's __init__ to set name/age
        self.marks = marks            # then add Student-specific data

    def average(self):
        return sum(self.marks) / len(self.marks)


student1 = Student("Ali", 22, [78, 85, 90])
print(student1.introduce())      # inherited from Person
print(student1.average())        # defined in Student

Hi, I'm Ali and I'm 22 years old.
84.33333333333333


### Key ideas
- `class Student(Person):` means "Student **is a** Person, with extra features."
- `super().__init__(name, age)` calls the **parent class's** constructor, so you don't have to repeat `self.name = name` and `self.age = age` again.

**Common mistake:** Forgetting to call `super().__init__(...)` in the child class — this means the parent's setup code never runs, and attributes like `self.name` may never get created.

<a id="overriding"></a>
# 10. Method Overriding and `super()`

A child class can **override** a parent's method — provide its own version with the same name. You can still call the parent's original version from inside the override using `super()`, if you want to extend rather than fully replace it.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"Hi, I'm {self.name}."


class Student(Person):
    def __init__(self, name, age, marks):
        super().__init__(name, age)
        self.marks = marks

    def introduce(self):
        # Override: extend the parent's version instead of replacing it entirely
        base_intro = super().introduce()
        return f"{base_intro} I'm a student with an average of {sum(self.marks)/len(self.marks):.2f}."


person1 = Person("Bilal", 30)
student1 = Student("Ali", 22, [78, 85, 90])

print(person1.introduce())    # Person's own version
print(student1.introduce())   # Student's overridden version

<a id="polymorphism"></a>
# 11. Polymorphism

**Polymorphism** ("many forms") means objects of different classes can be used through the **same interface** — the same method name behaves appropriately for each object's own type. You've actually already seen this above: calling `.introduce()` on a `Person` and a `Student` produced different results, without the calling code needing to know which exact class it was dealing with.

In [11]:
class Dog:
    def speak(self):
        return "Woof!"


class Cat:
    def speak(self):
        return "Meow!"


animals = [Dog(), Cat(), Dog()]

for animal in animals:
    print(animal.speak())   # same method call, different behavior per object

Woof!
Meow!
Woof!


### ML Connection
Scikit-learn relies heavily on polymorphism: `LinearRegression()`, `DecisionTreeClassifier()`, and `RandomForestClassifier()` are all *different* classes, but they all support the *same* `.fit(X, y)` and `.predict(X)` method names. This consistent interface is exactly why you can often swap one model for another with barely any code changes.

<a id="ml-preview"></a>
# 12. ML Preview: The `fit` / `predict` Pattern

Let's build a tiny, simplified class that mimics the *shape* of a real ML model's API — no real machine learning happens here, just the pattern. This is purely to make Scikit-learn feel familiar later, not a real algorithm.

In [12]:
class SimpleAverageModel:
    """
    A toy 'model' that just remembers the average of the training values
    and always predicts that average. This mimics the fit/predict shape
    used by real ML models, without doing any real machine learning.
    """

    def __init__(self):
        self.learned_average = None   # nothing learned yet

    def fit(self, training_values):
        self.learned_average = sum(training_values) / len(training_values)
        print(f"Model trained. Learned average: {self.learned_average:.2f}")

    def predict(self, number_of_predictions):
        if self.learned_average is None:
            raise ValueError("Model has not been trained yet. Call fit() first.")
        return [self.learned_average] * number_of_predictions


model = SimpleAverageModel()
model.fit([10, 20, 30, 40, 50])
predictions = model.predict(3)
print(predictions)

Model trained. Learned average: 30.00
[30.0, 30.0, 30.0]


In [13]:
# Calling predict() before fit() raises a clear, informative error -- a good practice to copy
untrained_model = SimpleAverageModel()

try:
    untrained_model.predict(3)
except ValueError as error:
    print("Error:", error)

Error: Model has not been trained yet. Call fit() first.


When you eventually write `model = LinearRegression(); model.fit(X_train, y_train); model.predict(X_test)` in Scikit-learn, you'll now understand exactly what's happening under the hood: an object is storing what it learned in `fit()` as internal attributes (like our `self.learned_average`), and using that stored state inside `predict()`.

<a id="common-mistakes"></a>
# 13. Common Mistakes — Recap

| Mistake | Fix |
|---|---|
| Forgetting `self` as the first parameter | Every instance method needs `self` first |
| Forgetting `super().__init__(...)` in a child class | Call it so the parent's setup logic actually runs |
| Using a mutable class attribute (e.g. a list) expecting per-object copies | Initialize mutable data inside `__init__` as `self.attribute = [...]` |
| Modifying a `_protected` attribute directly from outside | Prefer the class's own methods when they exist |
| Calling a method before required setup (like `predict()` before `fit()`) | Track state (e.g. `self.learned_average = None`) and raise a clear error |

<a id="summary"></a>
# Summary

- **Classes** are blueprints; **objects** are specific instances built from them.
- `__init__` sets up an object's initial data; `self` refers to the current object.
- **Instance attributes** are per-object; **class attributes** are shared.
- **Methods** are functions that live inside a class and use `self` to access the object's data.
- **Encapsulation** is managed in Python through naming conventions (`_protected`, `__private`).
- `__str__` controls how an object looks when printed.
- **Inheritance** lets a child class reuse and extend a parent class's behavior, using `super()`.
- **Polymorphism** lets different classes share the same method names, each with their own behavior — the exact pattern behind Scikit-learn's consistent `.fit()` / `.predict()` API.

**Next:** Open `OOP_Practice_Exercises.ipynb` to practice these concepts hands-on.